In [ ]:
import torch
print(torch.version.cuda)   
print(torch.__version__)     
print(torch.cuda.get_device_name(0)) 

In [ ]:
# Import the necessary libraries

import time
import os
import sys
import gdown
import terratorch
import albumentations
import lightning.pytorch as pl
import matplotlib.pyplot as plt
from pathlib import Path
from terratorch.datamodules import GenericNonGeoSegmentationDataModule
import warnings
# Suppress warnings
warnings.filterwarnings("ignore")

# Environment setup for TensorBoard proxy (assuming running in a notebook environment)
nb_prefix = os.environ.get("NB_PREFIX", "")
os.environ["TENSORBOARD_PROXY_URL"] = nb_prefix + "/proxy/6006/"

import torch
import torch.nn as nn
# Import the pruning utilities from torch.nn.utils
import torch.nn.utils.prune as prune
from torch.utils.data import Dataset, DataLoader
import numpy as np
from pathlib import Path
from PIL import Image

from albumentations.pytorch.transforms import ToTensorV2
import torch_pruning as tp
from tqdm import tqdm
from thop import profile, clever_format
import pytorch_lightning as pl
from fastcore.basics import patch_to
import torch.nn.functional as F
import timm
from terratorch.models.backbones.terramind.model.tm_utils import Attention as TerraMindAttention


from codecarbon import EmissionsTracker

## HLS Burn Scars Dataset

In [ ]:
# dataset_path = Path('hls_burn_scars')
# !ls "hls_burn_scars/"
dataset_path = Path('/burnScars/hls_burn_scars')
!ls "burnScars/hls_burn_scars"

In [ ]:
!ls {dataset_path}/data/ | head

In [ ]:
# Define the data module

datamodule = terratorch.datamodules.GenericNonGeoSegmentationDataModule(
    batch_size=8,
    num_workers=4,
    num_classes=2,


    # Define dataset paths 
    train_data_root=dataset_path / 'data/',
    train_label_data_root=dataset_path / 'data/',
    val_data_root=dataset_path / 'data/',
    val_label_data_root=dataset_path / 'data/',
    test_data_root=dataset_path / 'data/',
    test_label_data_root=dataset_path / 'data/',

    # Define splits
    train_split=dataset_path / 'peft-geofm-splits/train_data.txt',
    val_split=dataset_path / 'peft-geofm-splits/val_data.txt',
    test_split=dataset_path / 'peft-geofm-splits/test_data.txt',
    
    img_grep='*_merged.tif',
    label_grep='*.mask.tif',
    
    train_transform=[
        albumentations.D4(), # Random flips and rotation
        albumentations.pytorch.transforms.ToTensorV2(),
    ],

    transform = [
        albumentations.pytorch.transforms.ToTensorV2(),
    ],

    val_transform= [albumentations.pytorch.transforms.ToTensorV2(),],
    test_transform= [albumentations.pytorch.transforms.ToTensorV2(),],
        
    # standardization  for normalization
    means=[
      0.0333497067415863,
      0.0570118552053618,
      0.0588974813200132,
      0.2323245113436119,
      0.1972854853760658,
      0.1194491422518656,
    ],
    stds=[
      0.0226913556882377,
      0.0268075602230702,
      0.0400410984436278,
      0.0779173242367269,
      0.0870873883814014,
      0.0724197947743781,
    ],
    no_data_replace=0, #
    no_label_replace=-1,

)

# Setup train and val datasets
datamodule.setup("fit")

In [ ]:
# checking datasets train split size
train_dataset = datamodule.train_dataset
len(train_dataset)

In [ ]:
# checking datasets validation split size
val_dataset = datamodule.val_dataset
len(val_dataset)

In [ ]:
# checking datasets testing split size
datamodule.setup("test")
test_dataset = datamodule.test_dataset
len(test_dataset)

In [ ]:
# # plotting a few samples
# val_dataset.plot(val_dataset[9])
# val_dataset.plot(val_dataset[65])
# val_dataset.plot(val_dataset[100])

# L2 0.3 Pruning TerraMind-100 on BurnScars

In [ ]:
# -----------------------------
# Step 1: dynamic head_dim calculation
# -----------------------------
@patch_to(timm.models.vision_transformer.Attention)
def forward(self, x, attn_mask=None):
    B, N, C = x.shape
    qkv_out = self.qkv(x)
    qkv_dim = qkv_out.shape[-1]
    head_dim = max(1, qkv_dim // (3 * max(1, getattr(self, "num_heads", 1))))
    qkv = qkv_out.reshape(B, N, 3, getattr(self, "num_heads", 1), head_dim).permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    if hasattr(self, 'q_norm') and hasattr(self, 'k_norm'):
        q, k = self.q_norm(q), self.k_norm(k)
    scale = head_dim ** -0.5
    if getattr(self, "fused_attn", False):
        x = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.attn_drop.p if self.training else 0.0)
    else:
        q = q * scale
        attn = q @ k.transpose(-2, -1)
        if attn_mask is not None:
            attn = attn + attn_mask
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = attn @ v
    x = x.transpose(1, 2).reshape(B, N, -1)
    x = self.proj(x)
    x = self.proj_drop(x)
    return x

@patch_to(TerraMindAttention)
def forward(self, x, attn_mask=None):
    B, N, C = x.shape
    qkv_out = self.qkv(x)
    qkv_dim = qkv_out.shape[-1]
    head_dim = max(1, qkv_dim // (3 * max(1, getattr(self, "num_heads", 1))))
    qkv = qkv_out.reshape(B, N, 3, getattr(self, "num_heads", 1), head_dim).permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    if hasattr(self, 'q_norm') and hasattr(self, 'k_norm'):
        q, k = self.q_norm(q), self.k_norm(k)
    scale = head_dim ** -0.5
    if getattr(self, "fused_attn", False):
        x = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, dropout_p=self.attn_drop.p if self.training else 0.0)
    else:
        q = q * scale
        attn = q @ k.transpose(-2, -1)
        if attn_mask is not None:
            attn = attn + attn_mask
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)
        x = attn @ v
    x = x.transpose(1, 2).reshape(B, N, -1)
    x = self.proj(x)
    x = self.proj_drop(x)
    return x

print("✓ Patched Attention forward (timm + TerraMind) to compute head_dim dynamically")

In [ ]:
# ============================================================================
# STEP 2: Load Fine-tuned Model from Checkpoint for Pruning
# ============================================================================

checkpoint_path = "full_FT.ckpt"

# Check if checkpoint exists
from pathlib import Path
if not Path(checkpoint_path).exists():
    raise FileNotFoundError(f'Checkpoint not found: {checkpoint_path}')

print(f'Loading from: {checkpoint_path}')
print('\n=== Loading Fine-tuned Model for Pruning ===')

# Load using load_from_checkpoint (RECOMMENDED)
model = terratorch.tasks.SemanticSegmentationTask.load_from_checkpoint(
    checkpoint_path,
    model_factory="EncoderDecoderFactory",
    model_args={
        "backbone": "terramind_v1_base",
        "backbone_pretrained": False,
        # "backbone_img_size": 512,
        "backbone_modalities": ["S2L2A"],
        "backbone_bands": {
            "S2L2A": ["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"],
        },
        "necks": [
            {"name": "SelectIndices", 
             "indices":[2, 5, 8, 11]}, #  terramind_v1_base
            {"name": "ReshapeTokensToImage", "remove_cls_token": False},
            {"name": "LearnedInterpolateToPyramidal"}
        ],
        "decoder": "LinearDecoder",
        "decoder_upsampling_size": 16,
        "num_classes": 2,
    },
    loss="ce", 
    ignore_index=-1,
    # freeze_backbone=False,
    optimizer="AdamW",
    lr =4.264887762454439e-05,
    scheduler="ReduceLROnPlateau",
    scheduler_hparams={
        "patience": 4,
        "factor": 0.5,
    },
    class_names=["Unburned", "Burned"],
    plot_on_val=False, 
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
model.eval()

print(f'Fine-tuned model loaded from checkpoint on {device}')
print(f'Ready for pruning')

In [ ]:
# ============================================================================
# STEP 3: Verify the model loaded correctly
# ============================================================================
print('\n=== Model Verification ===')
example_input = torch.randn(4, 6, 512, 512).to(device) # based on image shape
with torch.no_grad():
    output = model(example_input)
    if hasattr(output, 'output'):
        print(f'✓ Output shape: {output.output.shape}')
    else:
        print(f'✓ Output shape: {output.shape}')

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params/1e6:.2f}M')
print(f'Trainable parameters: {trainable_params/1e6:.2f}M')


# Wrap model for pruning (extracts a single tensor)
class ModelWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        out = self.model(x)
        if isinstance(out, torch.Tensor):
            return out
        if hasattr(out, "output"):
            return out.output
        if hasattr(out, "__dict__"):
            for v in out.__dict__.values():
                if isinstance(v, torch.Tensor):
                    return v
        raise RuntimeError("Cannot extract tensor from model output")

wrapped_model = ModelWrapper(model.model if hasattr(model, "model") else model).to(device).eval()

# Use 224x224 example inputs for TerraMind-100
example_inputs = torch.randn(4, 6, 512, 512).to(device)
with torch.no_grad():
    sample = wrapped_model(example_inputs)
    print("Sample forward output shape:", getattr(sample, "shape", None))

In [ ]:
# -----------------------------
# STEP 4: Configure pruning
# -----------------------------
ignored_layers = []
ignored_names = []
unwrapped_parameters = []
num_heads = {}

# Build maps and detect modules
name_to_module = {n: m for n, m in wrapped_model.named_modules()}

for name, module in wrapped_model.named_modules():
    lname = name.lower()
    # Protect encoder embedding modules and patch embed modules
    if "encoder_embeddings" in lname or "patch_embed" in lname:
        ignored_layers.append(module)
        ignored_names.append(name)
    # Protect final single-output heads
    if isinstance(module, nn.Linear) and getattr(module, "out_features", None) == 1:
        ignored_layers.append(module)
        ignored_names.append(name)
    # Detect attention-like modules and populate num_heads reliably (key by qkv module if possible)
    if hasattr(module, "qkv") and hasattr(module, "num_heads"):
        qkv = getattr(module, "qkv")
        try:
            num_heads[qkv] = int(getattr(module, "num_heads"))
        except Exception:
            # fallback: key by module itself
            num_heads[module] = int(getattr(module, "num_heads"))


# Protect positional/modality/cls parameters (list of (param, dim))
for name, param in wrapped_model.named_parameters():
    lname = name.lower()
    if "pos_embed" in lname or "cls_token" in lname or "mod_emb" in lname or "modality" in lname:
        unwrapped_parameters.append((param, -1))

# dedupe ignored_layers
seen = set()
dedup_ignored = []
for m in ignored_layers:
    if id(m) not in seen:
        dedup_ignored.append(m)
        seen.add(id(m))
ignored_layers = dedup_ignored

print("Ignored layers count:", len(ignored_layers))
print("Example ignored names:", ignored_names[:10])
print("Unwrapped parameters count:", len(unwrapped_parameters))
print("Detected attention qkv modules:", len(num_heads))


In [ ]:
# --------------------------------------------
# STEP 6: Define pruning strategy and execute
# # ------------------------------------------
pruning_ratio = 0.3  # tune this to either 0.5, 0.7 or 0.9

pruner = tp.pruner.MetaPruner(
    model=wrapped_model,
    example_inputs=example_inputs,
    importance=tp.importance.MagnitudeImportance(p=2), # L2 norm importance
    global_pruning=True,
    pruning_ratio=pruning_ratio,
    iterative_steps=1,
    ignored_layers=ignored_layers,
    unwrapped_parameters=unwrapped_parameters,
    num_heads=num_heads,
    prune_num_heads=False,   
    prune_head_dims=False,   
    head_pruning_ratio=0.0,
    root_module_types=[nn.Linear, nn.Conv2d, nn.Conv3d],
    round_to=8,
    isomorphic=True,
)

print("Apply pruning")
pruner.step()
print("Pruning completed!")

In [ ]:
# --------------------------------
# STEP 7: Post-pruning processing
# --------------------------------

def fix_pos_emb_shape(enc):
    pos = enc.pos_emb
    D = pos.shape[-1]
    N = pos.shape[1] if pos.dim() == 3 else pos.shape[0]
    sqrt_N = int(N ** 0.5)
    target_N = sqrt_N * sqrt_N
    if N != target_N:
        # Interpolate or pad/truncate to nearest square
        pos_data = pos.data
        if pos_data.dim() == 3:
            pos_data = pos_data[0]
        pos_data = pos_data.permute(1, 0) if pos_data.shape[0] == D else pos_data
        pos_data = pos_data.unsqueeze(0).unsqueeze(0)  # [1, 1, N, D]
        pos_data = F.interpolate(pos_data, size=(target_N, D), mode='bilinear', align_corners=False)
        pos_data = pos_data.squeeze(0).squeeze(0)
        enc.pos_emb = nn.Parameter(pos_data.unsqueeze(0))
        print(f"Fixed pos_emb to shape: {enc.pos_emb.shape}")

# Usage: run for each encoder embedding after pruning
encoder_keys = model.model.encoder.encoder_embeddings
for key, enc in encoder_keys.items():
    fix_pos_emb_shape(enc)

print("Post-pruning processing completed!")

In [ ]:
# ============================================================================
# STEP 9: Setup Fine-tuning
# ============================================================================

print('\n=== Setting Up Fine-tuning ===')

pr = int(pruning_ratio * 100)

import lightning.pytorch as pl
import lightning.pytorch as lightningModule


checkpoint_callback = pl.callbacks.ModelCheckpoint(
    dirpath="output/TerraMind",
    mode="max",
    monitor="val/mIoU", 
    filename="L2_Prune",
)

# Lightning Trainer
trainer = pl.Trainer(
    accelerator="auto",
    strategy="auto",
    devices=1, #
    precision='bf16-mixed',  
    num_nodes=1,
    logger=True,  
    max_epochs=1, 
    log_every_n_steps=1,
    enable_checkpointing=True,
    callbacks=[checkpoint_callback, 
               pl.callbacks.RichProgressBar(),
               pl.callbacks.LearningRateMonitor(logging_interval="epoch"),
               pl.callbacks.EarlyStopping(monitor="val/loss", patience=30),
              ],
    default_root_dir="output/",
    detect_anomaly=True,
)

print('✓ Trainer configured')

In [ ]:
# ============================================================================
# STEP 10: Start Fine-tuning
# ============================================================================

# Set to training mode
model.train()

start_time = time.time()

# try:
    # Fine-tune the pruned model directly
trainer.fit(model, datamodule=datamodule)
    
elapsed = time.time() - start_time
    
print('Fine-tuning Completed!')
print(f'Duration: {elapsed/60:.2f} minutes')
print(f'Epochs completed: {trainer.current_epoch}')
print(f'Best validation mIoU: {checkpoint_callback.best_model_score:.4f}')
    

In [ ]:
best_ckpt_path = checkpoint_callback.best_model_path

def run_test_and_track_metrics(model, datamodule, trainer, ckpt_path):
    """
    Runs model testing using a Lightning Trainer while tracking
    Carbon Emissions and measuring Inference Speed.
    """
    
    ckpt_name = Path(ckpt_path).stem
    
    # Carbon tracker
    tracker = EmissionsTracker(
        project_name=ckpt_name, 
        log_level="error" # Minimize console output
    )
    tracker.start()
    
    # --- Test Evaluation ---
    start_infer = time.time()
    test_results = trainer.test(model, datamodule=datamodule, ckpt_path=ckpt_path)
        # test_results = [{}]
    end_infer = time.time()
    
    # Stop carbin tracker 
    emissions_kg = tracker.stop() # Returns the total measured emissions
    energy = tracker._total_energy.kWh
    infer_duration = end_infer - start_infer
    
    # Calculate Inference Speed (Throughput)
    test_loader = datamodule.test_dataloader()
    num_samples = len(test_loader.dataset)
    throughput = num_samples / infer_duration if infer_duration > 0 else 0
    
    # Summary report
    print('\n' + '='*50)
    print(f'✅ EVALUATION SUMMARY: {ckpt_name}')
    print(f'Inference Time: {infer_duration:.2f} s')
    print(f'Throughput: {throughput:.2f} samples/s')
    print(f'Measured Carbon: {emissions_kg:.6f} kg CO₂eq')
    print(f'Measured Energy: {energy:.6f} kWh')
    

# Inference and metrics tracking
Inference_output = run_test_and_track_metrics(model, datamodule, trainer, best_ckpt_path)